# Consumer Shopping Trends — Purchase Amount Prediction

**Goal:** Reproduce a customer-spending regression workflow on a different publicly available consumer-shopping dataset.

This notebook covers dataset download, inspection, data cleaning, EDA, simple linear regression, multiple linear regression, benchmark models and evaluation. Run cells from top to bottom.

In [ ]:
# Install dependencies if running in a fresh Google Colab/Jupyter environment
%pip install -q pandas numpy matplotlib seaborn scikit-learn joblib requests

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if (ROOT / "src").exists() is False and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
# Download the original public CSV. This is the real source dataset, not generated sample data.
import subprocess
subprocess.run([sys.executable, str(ROOT / "download_dataset.py")], check=True)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_utils import load_data, resolve_target, prepare_features

df = load_data()
display(df.head())
print("Shape:", df.shape)
print("\\nData types:")
display(df.dtypes.to_frame("dtype"))
print("\\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing"))
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
target = resolve_target(df)
print("Prediction target:", target)
display(df.describe(include="all").T)

## Exploratory Data Analysis

In [ ]:
# Target distribution
plt.figure(figsize=(8, 5))
sns.histplot(df[target].dropna(), kde=True)
plt.title(f"Distribution of {target}")
plt.xlabel(target)
plt.tight_layout()
plt.show()

In [ ]:
# Correlations among numeric variables
numeric_df = df.select_dtypes(include=np.number)
plt.figure(figsize=(12, 8))
sns.heatmap(numeric_df.corr(), cmap="coolwarm", center=0)
plt.title("Correlation Heatmap — Numeric Features")
plt.tight_layout()
plt.show()

In [ ]:
# Prepare modelling table: deduplicate, validate target, exclude identifiers and unsuitable columns.
X, y, clean = prepare_features(df, target)
print("Clean rows:", len(clean))
print("Predictor columns:", list(X.columns))
display(X.head())
print(y.describe())

## Train/test split and preprocessing

The test set is held out before fitting imputers, scalers or encoders. This prevents preprocessing leakage.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

def make_preprocessor(frame):
    numeric = frame.select_dtypes(include=np.number).columns.tolist()
    categorical = frame.select_dtypes(exclude=np.number).columns.tolist()
    num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),
                         ("scaler", StandardScaler())])
    cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                         ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    return ColumnTransformer([("num", num_pipe, numeric), ("cat", cat_pipe, categorical)])

print("Training rows:", len(X_train), "Testing rows:", len(X_test))

## Simple Linear Regression

In [ ]:
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
if not numeric_cols:
    raise ValueError("No numeric feature available for simple linear regression.")
correlations = X_train[numeric_cols].corrwith(y_train).abs().dropna().sort_values(ascending=False)
simple_feature = correlations.index[0]
simple_model = Pipeline([
    ("preprocessor", make_preprocessor(X_train[[simple_feature]])),
    ("model", LinearRegression())
])
simple_model.fit(X_train[[simple_feature]], y_train)
simple_pred = simple_model.predict(X_test[[simple_feature]])
print("Selected single predictor:", simple_feature)

## Multiple Linear Regression

In [ ]:
multiple_model = Pipeline([
    ("preprocessor", make_preprocessor(X_train)),
    ("model", LinearRegression())
])
multiple_model.fit(X_train, y_train)
multiple_pred = multiple_model.predict(X_test)

## Additional model benchmarks

In [ ]:
ridge_model = Pipeline([
    ("preprocessor", make_preprocessor(X_train)),
    ("model", Ridge(alpha=1.0))
])
ridge_model.fit(X_train, y_train)
ridge_pred = ridge_model.predict(X_test)

rf_model = Pipeline([
    ("preprocessor", make_preprocessor(X_train)),
    ("model", RandomForestRegressor(n_estimators=250, min_samples_leaf=2,
                                    random_state=42, n_jobs=-1))
])
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

In [ ]:
def evaluate(y_true, pred):
    return {
        "MAE": mean_absolute_error(y_true, pred),
        "MSE": mean_squared_error(y_true, pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, pred)),
        "R2": r2_score(y_true, pred)
    }

results = pd.DataFrame([
    {"Model": "Simple Linear Regression", **evaluate(y_test, simple_pred)},
    {"Model": "Multiple Linear Regression", **evaluate(y_test, multiple_pred)},
    {"Model": "Ridge Regression", **evaluate(y_test, ridge_pred)},
    {"Model": "Random Forest Regressor", **evaluate(y_test, rf_pred)}
]).sort_values("RMSE")
display(results)
results.to_csv(ROOT / "outputs" / "model_comparison.csv", index=False)

## Evaluation visualizations

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(x=y_test, y=multiple_pred, alpha=.65)
lo = min(float(y_test.min()), float(multiple_pred.min()))
hi = max(float(y_test.max()), float(multiple_pred.max()))
plt.plot([lo, hi], [lo, hi], "--")
plt.xlabel("Actual spending")
plt.ylabel("Predicted spending")
plt.title("Actual vs Predicted — Multiple Linear Regression")
plt.tight_layout()
plt.show()

In [ ]:
residuals = y_test.to_numpy() - multiple_pred
plt.figure(figsize=(7, 5))
sns.histplot(residuals, kde=True)
plt.title("Residual Distribution")
plt.xlabel("Actual − Predicted")
plt.tight_layout()
plt.show()

## Predict for a new customer

The example below uses median values for numeric columns and the most frequent category for categorical columns. Replace these values with a valid customer profile to test another scenario.

In [ ]:
new_customer = {}
for col in X.columns:
    if pd.api.types.is_numeric_dtype(X[col]):
        new_customer[col] = float(X[col].median()) if X[col].notna().any() else 0.0
    else:
        modes = X[col].dropna().astype(str).mode()
        new_customer[col] = modes.iloc[0] if len(modes) else "Unknown"

new_customer_df = pd.DataFrame([new_customer])
estimated_spend = multiple_model.predict(new_customer_df)[0]
print("Example estimated", target, "=", round(float(estimated_spend), 2))

## Conclusion and interpretation

- Compare models using held-out test metrics rather than training performance.
- Lower MAE/RMSE indicate smaller errors in the target's units; higher R² indicates more explained variance.
- Simple Linear Regression uses one numeric predictor, while Multiple Linear Regression uses all eligible features.
- Ridge and Random Forest are comparison models, not guaranteed improvements.
- Interpret results in the context of the source data and its collection methodology.

**Reproducibility:** fixed random seed = 42; test size = 20%; preprocessing is fitted only on training data.